# Benchpark: Expert Evaluation of LLMs

This notebook presents a practical workflow to evaluate LLM responses with expert judges.

Each LLM alert is scored from **1 to 5** on three key metrics, then results are aggregated to compare models or different model versions.

## Evaluation rubric (1-5 scale)

| Metric | Description | 1 (Poor) | 5 (Excellent) |
|---|---|---|---|
| Factuality | Is the information truthful and free of hallucinations? | Contains serious factual errors. | 100% truthful and verifiable. |
| Instruction Following | Does it follow format, tone, and constraints? | Ignored the instructions. | Followed every requirement. |
| Helpfulness | How much value does it provide to the end user? | Useless or needs full rework. | Really relevant alert. |

In [1]:
from pathlib import Path
import json
import sys
import pandas as pd
from ast import literal_eval
from IPython.display import Markdown, display
from pipeline import NewsAlertPipeline
import numpy as np
import pingouin as pg

c:\Users\lucia\anaconda3\envs\deep_new\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
METRICS = [
    (
        "factuality",
        "Does the alert stay faithful to the article and avoid hallucinations?",
    ),
    (
        "instruction_following",
        "Does the alert follow the expected format, tone, and task constraints?",
    ),
    (
        "helpfulness",
        "How useful is the alert for a real evaluator or end user?",
    ),
]
METRICS = ["factuality", "instruction_following", "helpfulness"]
EXPERT_COLUMNS = ["lucia", "alvaro", "beatriz", "alberto"]
NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "pipeline.py").exists() and (NOTEBOOK_DIR / "src" / "pipeline.py").exists():
    NOTEBOOK_DIR = NOTEBOOK_DIR / "src"
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

PROJECT_DIR = NOTEBOOK_DIR.parent if (
    NOTEBOOK_DIR / "pipeline.py").exists() else NOTEBOOK_DIR.parent
DATA_DIR = PROJECT_DIR / "data"

##### Helper Functions

In [3]:
def _parse_scores(value):
    if isinstance(value, list):
        return value
    if pd.isna(value):
        return None
    if isinstance(value, str):
        text = value.strip()
        if not text:
            return None
        try:
            parsed = json.loads(text)
        except json.JSONDecodeError:
            parsed = literal_eval(text)
        return list(parsed) if isinstance(parsed, (list, tuple)) else None
    return None

def _to_optional_text(value):
    if pd.isna(value):
        return None
    text = str(value).strip()
    return text or None

# Main Pipeline

### Generate alerts
In the following cell we generate alerts on the selected articles using our alert model.

In [ ]:
EXPERT_IMAGES_DIR = DATA_DIR / "expert_images"
INPUT_PATH = DATA_DIR / "articles_expert_benchmark.csv"
OUTPUT_PATH = DATA_DIR / "articles_expert_benchmark_with_alerts.csv"


In [ ]:

benchmark_df = pd.read_csv(INPUT_PATH, sep=";")
required_columns = ["article_text", "image_path"]
missing_columns = [
    column for column in required_columns if column not in benchmark_df.columns]
if missing_columns:
    raise ValueError(
        f"Missing required columns in {INPUT_PATH.name}: {missing_columns}")

pipeline = NewsAlertPipeline()


def _resolve_image_path(raw_path):
    if not raw_path:
        return None
    candidate = Path(raw_path)
    if candidate.is_absolute() and candidate.exists():
        return str(candidate)
    root_relative = PROJECT_DIR / candidate
    if root_relative.exists():
        return str(root_relative)
    image_dir_relative = EXPERT_IMAGES_DIR / candidate
    if image_dir_relative.exists():
        return str(image_dir_relative)
    return raw_path

def _serialize_alert(alert):
    return json.dumps(alert, ensure_ascii=False) if isinstance(alert, (dict, list)) else alert


if "alert_generated" not in benchmark_df.columns:
    benchmark_df["alert_generated"] = pd.NA

for row_index, row in benchmark_df.iterrows():
    article_text = _to_optional_text(row["article_text"])
    image_path = _resolve_image_path(_to_optional_text(row["image_path"]))
    if not article_text:
        raise ValueError(f"Row {row_index} has an empty article_text.")
    result = pipeline.run(article_text=article_text, image_path=image_path)
    benchmark_df.at[row_index, "alert_generated"] = _serialize_alert(
        result["final_alert"])

benchmark_df.to_csv(OUTPUT_PATH, index=False, sep=";")
benchmark_df

### Expert Input

The next sections runs a script to facilitate the score assignation for each article and metric for the expert.
The expert needs to input their name correctly and give a score for each article sequentially.

In [ ]:
EXPERT_COLUMNS = ["lucia", "alvaro", "beatriz", "alberto"]
INPUT_PATH = DATA_DIR / "articles_expert_benchmark_with_alerts.csv"
OUTPUT_PATH = DATA_DIR / "articles_expert_benchmark_with_expert_scores.csv"

benchmark_df = pd.read_csv(INPUT_PATH, sep=";")
for column in EXPERT_COLUMNS:
    if column not in benchmark_df.columns:
        benchmark_df[column] = pd.NA

def _save_progress(frame):
    frame.to_csv(OUTPUT_PATH, index=False, sep=";")


expert_name = input("Expert name (lucia, alvaro, beatriz, alberto): ").lower()
if expert_name not in EXPERT_COLUMNS:
    raise ValueError(f"Unknown expert '{expert_name}'. Use one of: {EXPERT_COLUMNS}")

print(f"Recording scores for: {expert_name}")
print(f"Progress will be saved to: {OUTPUT_PATH}")

for row_index, row in benchmark_df.iterrows():
    existing_scores = _parse_scores(row[expert_name])
    if isinstance(existing_scores, list) and len(existing_scores) == len(METRICS):
        continue

    print("\n" + "=" * 90)
    print(f"Article {row_index + 1}/{len(benchmark_df)}")
    print("=" * 90)
    display(Markdown(f"**Article text:**\n\n{row['article_text']}"))

    image_path = row.get("image_path")
    if isinstance(image_path, str) and image_path.strip():
        print(f"Image path: {image_path}")

    if pd.notna(row.get("alert_generated")):
        print(f"Generated alert: {row['alert_generated']}")

    scores = []
    for metric_name, metric_description in METRICS:
        while True:
            print(f"\nMetric: {metric_name}")
            print(metric_description)
            score_text = input("What score do you give this metric? 1-5: ").strip()
            try:
                score = int(score_text)
            except ValueError:
                print("Enter an integer between 1 and 5.")
                continue
            if 1 <= score <= 5:
                scores.append(score)
                break
            print("Enter an integer between 1 and 5.")

    benchmark_df.at[row_index, expert_name] = scores
    _save_progress(benchmark_df)
    print(f"Saved {expert_name} scores for row {row_index + 1}: {scores}")

benchmark_df

Recording scores for: beatriz
Progress will be saved to: /workspaces/final-project-nlp-dl-gru/data/articles_expert_benchmark_with_expert_scores.csv

Article 1/20


**Article text:**

U.S. President Donald Trump announced this Sunday that he is sending a negotiating team to Islamabad to hold a second meeting with Iranian officials before the truce agreed with Tehran expires this Wednesday. The president described the meeting as the last opportunity to accept peace and warned on social media that, if an agreement is not signed, the country would be bombed in key infrastructure such as bridges and power plants. This escalation in tension comes amid cross-threats over control of the Strait of Hormuz. Washington accuses Iran of attacking British and French cargo ships in recent hours, while Tehran claims it has only intercepted oil tankers to pressure against the U.S. blockade. Iran's army chief, Amir Hatami, responded to the warnings by saying his forces are ready for confrontation and are keeping their finger on the trigger in case armed conflict resumes. The U.S. delegation is scheduled to arrive in the Pakistani capital this Monday. Although Trump said Vice President JD Vance would not travel for security reasons, several U.S. and Pakistani media outlets claim that Vance will lead the team together with mediator Steve Witkoff and adviser Jared Kushner. Meanwhile, Iran's official Tasnim agency reported that Tehran will not attend talks while Washington's maritime blockade remains in place, although the speaker of the Iranian parliament, Mohammad Bagher Qalibaf, acknowledged that there has been some progress, even if a final agreement still appears distant. The nuclear program remains the second major obstacle in the negotiations. The Trump administration is demanding full dismantlement and the handover of all enriched uranium, while the Islamic Republic insists on maintaining its enrichment plan for civilian use. International analysts warn that the fragility of the open fronts in Hormuz, the nuclear sphere, and the offensive in Lebanon could cause the diplomatic process to fail at any moment.

Image path: ../data/expert_images/trump_iran_reunion.png
Generated alert: {"is_alert": true, "alert_level": "critical", "category": "geopolitical", "reason": "The U.S. President's announcement of sending a negotiating team to Islamabad to hold a second meeting with Iranian officials before the truce expires, along with the cross-threats over control of the Strait of Hormuz, indicates a high level of geopolitical risk and potential for escalation.", "source": "fine_tuned_alert_model"}

Metric: factuality
Does the alert stay faithful to the article and avoid hallucinations?

Metric: instruction_following
Does the alert follow the expected format, tone, and task constraints?

Metric: helpfulness
How useful is the alert for a real evaluator or end user?
Saved beatriz scores for row 1: [5, 5, 5]

Article 2/20


**Article text:**

Within the final phase of the truce in the conflict involving the United States, Israel, and Iran, President Donald Trump reported this Monday that the U.S. Navy intercepted and forcibly seized an Iranian-flagged cargo ship. According to the White House version, the vessel was captured after attempting to evade the naval blockade imposed by Washington near the Strait of Hormuz. As an immediate response to this action, Iran blocked the passage of two oil tankers in the same area, while the president of the Iranian parliament warned that his country's forces will maintain the blockade of this strategic route for the global crude trade as long as the free movement of their own ships is prevented.

Image path: ../data/expert_images/iran_boat.png
Generated alert: {"is_alert": true, "alert_level": "critical", "category": "geopolitical", "reason": "The U.S. Navy intercepted and forcibly seized an Iranian-flagged cargo ship, leading to Iran blocking the passage of two oil tankers in the same area and warning that their forces will maintain the blockade of the strategic route for the global crude trade as long as the free movement of their own ships is prevented.", "source": "fine_tuned_alert_model"}

Metric: factuality
Does the alert stay faithful to the article and avoid hallucinations?

Metric: instruction_following
Does the alert follow the expected format, tone, and task constraints?

Metric: helpfulness
How useful is the alert for a real evaluator or end user?
Saved beatriz scores for row 2: [5, 5, 5]

Article 3/20


**Article text:**

Optometrist Alberto Villarroya explains how the color change in operating rooms, from traditional white to green and blue, has been fundamental to improving surgeons' precision and patient safety. Historically, white dominated medical environments because of its association with cleanliness, but practice showed that this color was counterproductive during procedures due to a visual phenomenon known as afterimage. As the specialist details, when a surgeon spends a long time looking at the intense red color of blood, the eye cells responsible for perceiving that color become fatigued. If the professional then looks away at a white surface, the brain tries to compensate for that fatigue by generating a residual image in green or bluish tones. This visual distortion is not only annoying, but also causes eye strain and makes focusing more difficult, factors that can be critical in long and complex operations. The solution was to replace white with green or blue in gowns, walls, and surgical drapes, which helps neutralize the effect of visual fatigue and preserve the surgeon's sharpness. This adjustment, now a global standard in hospitals, has improved medical staff concentration and reduced risks, showing how a technical detail in the work environment can directly influence the success of interventions.

... (32 outputs left out)

Image path: ../data/expert_images/seafood.png
Generated alert: {"is_alert": false, "alert_level": null, "category": null, "reason": null, "source": "fine_tuned_alert_model"}

Metric: factuality
Does the alert stay faithful to the article and avoid hallucinations?

Metric: instruction_following
Does the alert follow the expected format, tone, and task constraints?

Metric: helpfulness
How useful is the alert for a real evaluator or end user?
Saved beatriz scores for row 19: [5, 5, 5]

Article 20/20


**Article text:**

Dyson has launched the HushJet Mini Cool, a new $99.99 handheld fan designed with a slim, cylindrical body that measures 38mm in diameter and weighs 208 grams. This weight is comparable to an iPhone 17 Pro, making the device highly portable and easy to slip into a pocket. Unlike many competing models that feature bulky tops or exposed blades, Dyson’s version contains all components—including the motor, spinning blades, and a 5,000mAh rechargeable battery—entirely within its sleek chassis. The device is operated via a toggle switch for power and a rocker button that cycles through five speed levels, which are indicated by five small white LEDs. Additionally, a temporary Boost Mode can be activated for maximum airflow by holding down the rocker button. While the fan is noted for being significantly more powerful than other pocketable alternatives, it has been criticized for a noise level that contradicts its "Hush" branding, which may make it less suitable for use in quiet environments such as ceremonies or formal receptions despite its effectiveness in hot and humid weather.

Image path: ../data/expert_images/dyson.png
Generated alert: {"is_alert": false, "alert_level": null, "category": null, "reason": null, "source": "fine_tuned_alert_model"}

Metric: factuality
Does the alert stay faithful to the article and avoid hallucinations?

Metric: instruction_following
Does the alert follow the expected format, tone, and task constraints?

Metric: helpfulness
How useful is the alert for a real evaluator or end user?
Saved beatriz scores for row 20: [5, 5, 5]


,article_text,image_path,alert_generated,lucia,alvaro,beatriz,alberto
0,U.S. President Donald Trump announced this Sun...,../data/expert_images/trump_iran_reunion.png,"{""is_alert"": true, ""alert_level"": ""critical"", ...",<NA>,<NA>,"[5, 5, 5]",<NA>
1,Within the final phase of the truce in the con...,../data/expert_images/iran_boat.png,"{""is_alert"": true, ""alert_level"": ""critical"", ...",<NA>,<NA>,"[5, 5, 5]",<NA>
2,Optometrist Alberto Villarroya explains how th...,../data/expert_images/hombre_tik_tok.png,"{""is_alert"": false, ""alert_level"": null, ""cate...",<NA>,<NA>,"[5, 5, 5]",<NA>
3,"Richard DeFreece, a U.S. citizen from Texas, h...",../data/expert_images/aviones_papel.png,"{""is_alert"": false, ""alert_level"": null, ""cate...",<NA>,<NA>,"[5, 5, 5]",<NA>
4,The Al Musanaa camel beauty festival in Oman d...,../data/expert_images/camellos.png,"{""is_alert"": true, ""alert_level"": ""critical"", ...",<NA>,<NA>,"[3, 4, 2]",<NA>
5,"Helmut Marko, Red Bull adviser, has clarified ...",../data/expert_images/franco_colapinto.png,"{""is_alert"": false, ""alert_level"": null, ""cate...",<NA>,<NA>,"[5, 5, 2]",<NA>
6,Bitcoin suffered its worst price drop since 20...,../data/expert_images/bitcoin.png,"{""is_alert"": true, ""alert_level"": ""critical"", ...",<NA>,<NA>,"[4, 5, 5]",<NA>
7,Pharmacist Boticaria Garcia has warned about t...,../data/expert_images/enfermedades.png,"{""is_alert"": true, ""alert_level"": ""critical"", ...",<NA>,<NA>,"[4, 5, 3]",<NA>
8,Aena has announced its intention to finance th...,../data/expert_images/zumos.png,"{""is_alert"": false, ""alert_level"": null, ""cate...",<NA>,<NA>,"[2, 2, 1]",<NA>
9,Timmy the humpback whale has managed to free i...,../data/expert_images/ballena.png,"{""is_alert"": false, ""alert_level"": null, ""cate...",<NA>,<NA>,"[5, 5, 5]",<NA>


### Expert Score Validation

If the scores assigned by the experts are significantly different, then the conclusions extracted from this analysis might be skewed or biased. In order to avoid that, we are going to measure the similarity in the scores given by the experts. If they are significantly similar, we can carry on with the study, if not, we need to calibrate the expert judgement.

In [15]:
INPUT_PATH = DATA_DIR / "articles_expert_benchmark_with_expert_scores.csv"
analysis_df = pd.read_csv(INPUT_PATH, sep=";")

def _build_icc_long_frame(frame, metric_index):
    rows = []
    for article_id, (_, row) in enumerate(frame.iterrows()):
        for expert_column in EXPERT_COLUMNS:
            scores = _parse_scores(row.get(expert_column))
            if scores is None or len(scores) <= metric_index:
                score_value = np.nan
            else:
                score_value = pd.to_numeric(scores[metric_index], errors="coerce")
            rows.append(
                {
                    "article": article_id,
                    "expert": expert_column,
                    "score": score_value,
                }
            )
    return pd.DataFrame(rows)


def _select_icc_row(icc_results):
    if icc_results is None or icc_results.empty:
        return None
    if "Type" not in icc_results.columns:
        return icc_results.iloc[0]

    preferred_types = ["ICC3K", "ICC2K", "ICC1K", "ICC3", "ICC2", "ICC1"]
    type_norm = icc_results["Type"].astype(str).str.strip().str.upper()
    for icc_type in preferred_types:
        mask = type_norm == icc_type
        if mask.any():
            return icc_results.loc[mask].iloc[0]
    return icc_results.iloc[0]


results_icc = []
for metric_index, metric_name in enumerate(METRICS):
    df_long = _build_icc_long_frame(analysis_df, metric_index)
    df_non_null = df_long.dropna(subset=["score"])
    n_raters = int(df_non_null["expert"].nunique())
    n_targets_total = int(df_non_null["article"].nunique())

    if n_raters < 2 or n_targets_total < 2:
        print(
            f"Skipping '{metric_name}': need at least 2 experts and 2 scored articles."
        )
        continue
    try:
        icc_results = pg.intraclass_corr(
            data=df_long,
            targets="article",
            raters="expert",
            ratings="score",
            nan_policy="omit",
        )

        selected = _select_icc_row(icc_results)
        if selected is None:
            print(f"Could not compute a valid ICC row for '{metric_name}'.")
            continue
        icc_value = pd.to_numeric(selected.get("ICC", np.nan), errors="coerce")
        if pd.isna(icc_value):
            print(f"Could not compute a numeric ICC value for '{metric_name}'.")
            continue

        results_icc.append(
            {
                "metric": metric_name,
                "icc_type": str(selected.get("Type", "unknown")),
                "icc_value": round(float(icc_value), 4),
                "n_samples": n_targets_total,
                "n_raters": n_raters,
                "experts_used": EXPERT_COLUMNS,
                "status": "Calculated",
            }
        )
    except Exception as e:
        print(f"Error computing ICC for {metric_name}: {e}")

df_icc_final = pd.DataFrame(results_icc)

if not df_icc_final.empty:
    print("\n--- Intraclass Correlation Coefficient (ICC) results ---")
    display(df_icc_final)
else:
    print("No ICC metrics could be computed. Check score completeness and parsing format.")


--- Intraclass Correlation Coefficient (ICC) results ---


,metric,icc_type,icc_value,n_samples,n_raters,experts_used,status
0,factuality,"ICC(1,1)",0.8899,20,4,"[lucia, alvaro, beatriz, alberto]",Calculated
1,instruction_following,"ICC(1,1)",0.9127,20,4,"[lucia, alvaro, beatriz, alberto]",Calculated
2,helpfulness,"ICC(1,1)",0.8665,20,4,"[lucia, alvaro, beatriz, alberto]",Calculated


## 4) Model score analysis

We use this section to inspect the model output against the expert ratings, compare the metrics, and surface the examples where the benchmark is most informative.

In [16]:
INPUT_PATH = DATA_DIR / "articles_expert_benchmark_with_expert_scores.csv"
METRICS = ["factuality", "instruction_following", "helpfulness"]
EXPERT_COLUMNS = ["lucia", "alvaro", "beatriz", "alberto"]
analysis_df = pd.read_csv(INPUT_PATH, sep=";")

def _parse_alert(value):
    if pd.isna(value):
        return {}
    if isinstance(value, dict):
        return value
    if isinstance(value, str):
        text = value.strip()
        if not text:
            return {}
        try:
            return json.loads(text)
        except json.JSONDecodeError:
            try:
                return literal_eval(text)
            except (ValueError, SyntaxError):
                return {"raw": value}
    return {"raw": value}


def _metric_value(scores, metric_index):
    if not isinstance(scores, list) or len(scores) <= metric_index:
        return pd.NA
    return scores[metric_index]


for metric_index, metric_name in enumerate(METRICS):
    for expert_column in EXPERT_COLUMNS:
        analysis_df[f"{expert_column}_{metric_name}"] = analysis_df[expert_column].apply(
            lambda value, idx=metric_index: _metric_value(_parse_scores(value), idx)
        )

    metric_expert_columns = [f"{expert_column}_{metric_name}" for expert_column in EXPERT_COLUMNS]
    analysis_df[metric_expert_columns] = analysis_df[metric_expert_columns].apply(
        pd.to_numeric, errors="coerce"
    )
    analysis_df[f"{metric_name}_mean"] = analysis_df[metric_expert_columns].mean(axis=1)
    analysis_df[f"{metric_name}_std"] = analysis_df[metric_expert_columns].std(axis=1)

analysis_df["alert_parsed"] = analysis_df["alert_generated"].apply(_parse_alert)
analysis_df["alert_is_alert"] = analysis_df["alert_parsed"].apply(
    lambda value: value.get("is_alert") if isinstance(value, dict) else None
)
analysis_df["alert_level"] = analysis_df["alert_parsed"].apply(
    lambda value: value.get("alert_level") if isinstance(value, dict) else None
)
analysis_df["overall_expert_mean"] = analysis_df[[f"{metric}_mean" for metric in METRICS]].mean(axis=1)
analysis_df["overall_disagreement"] = analysis_df[[f"{metric}_std" for metric in METRICS]].mean(axis=1)

score_columns = [f"{metric}_mean" for metric in METRICS]
summary = analysis_df[score_columns].agg(["mean", "std", "min", "max"]).T.round(2)
expert_summary = pd.DataFrame(
    {
        expert: [analysis_df[f"{expert}_{metric}"].mean() for metric in METRICS]
        for expert in EXPERT_COLUMNS
    },
    index=METRICS,
).round(2)

top_disagreement = analysis_df.sort_values("overall_disagreement", ascending=False)[
    ["article_text", "image_path", "alert_generated"] + score_columns + ["overall_disagreement"]
].head(5)

alert_comparison = analysis_df.groupby("alert_is_alert")[score_columns].mean().round(2)
metric_correlation = analysis_df[score_columns].corr().round(2)
alert_level_summary = analysis_df.groupby("alert_level")[score_columns].mean().round(2)

print("Average expert score per metric:")
display(summary)

print("Average score by expert:")
display(expert_summary)

print("Score comparison by model alert flag:")
display(alert_comparison)

print("Score comparison by alert level:")
display(alert_level_summary)

print("Correlation between metric averages:")
display(metric_correlation)

print("Top disagreement examples:")
display(top_disagreement)

Average expert score per metric:


,mean,std,min,max
factuality_mean,4.82,0.37,4.0,5.0
instruction_following_mean,4.86,0.56,2.5,5.0
helpfulness_mean,3.98,1.48,1.0,5.0


Average score by expert:


,lucia,alvaro,beatriz,alberto
factuality,4.85,4.80,4.80,4.85
instruction_following,4.90,4.90,4.85,4.80
helpfulness,4.05,4.15,3.85,3.85


Score comparison by model alert flag:


,factuality_mean,instruction_following_mean,helpfulness_mean
alert_is_alert,,,
False,5.00,4.72,4.78
True,4.68,4.98,3.32


... (2 outputs left out)

Correlation between metric averages:


,factuality_mean,instruction_following_mean,helpfulness_mean
factuality_mean,1.00,-0.06,0.31
instruction_following_mean,-0.06,1.00,-0.01
helpfulness_mean,0.31,-0.01,1.00


Top disagreement examples:


,article_text,image_path,alert_generated,factuality_mean,instruction_following_mean,helpfulness_mean,overall_disagreement
8,Aena has announced its intention to finance th...,../data/expert_images/zumos.png,"{""is_alert"": false, ""alert_level"": null, ""cate...",5.0,2.50,4.25,0.692450
5,"Helmut Marko, Red Bull adviser, has clarified ...",../data/expert_images/franco_colapinto.png,"{""is_alert"": false, ""alert_level"": null, ""cate...",5.0,5.00,4.00,0.471405
4,The Al Musanaa camel beauty festival in Oman d...,../data/expert_images/camellos.png,"{""is_alert"": true, ""alert_level"": ""critical"", ...",4.5,5.00,1.25,0.359117
7,Pharmacist Boticaria Garcia has warned about t...,../data/expert_images/enfermedades.png,"{""is_alert"": true, ""alert_level"": ""critical"", ...",4.0,4.75,1.75,0.333333
16,"A domestic violence tragedy in Shreveport, Lou...",../data/expert_images/policia.png,"{""is_alert"": true, ""alert_level"": ""critical"", ...",5.0,5.00,4.00,0.272166
